In [ ]:
import os
from google.colab import drive
from datasets import load_from_disk
from dotenv import load_dotenv
import numpy as np
from PIL import Image
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torchmetrics
import piq
from typing import Dict

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
TRAIN_DENOISER=False
DATASET_DIR = "/content/drive/MyDrive/AAI-521 Final Project/denoiser" if (TRAIN_DENOISER) else "/content/drive/MyDrive/AAI-521 Final Project/super_res"
os.makedirs(DATASET_DIR, exist_ok=True)


In [ ]:
env_path = "/content/drive/MyDrive/AAI-521 Final Project/hf_login.env"
# Open env file
load_dotenv(env_path)
# Access token from environment
hf_key = os.getenv("HF_TOKEN")
print("HF key loaded:", hf_key is not None)

HF key loaded: True


In [ ]:
from datasets import load_dataset

#train = load_dataset("detection-datasets/coco", split="train[:200]")
#val   = load_dataset("detection-datasets/coco", split="train[200:250]")
#test  = load_dataset("detection-datasets/coco", split="train[250:300]")

# the dataset is in denoiser dir, future: move to coco_dataset
def load_or_create_coco_dataset():
    if os.path.exists(DATASET_DIR):
        print(f"📂 Loading dataset from disk: {DATASET_DIR}")
        return load_from_disk("/content/drive/MyDrive/AAI-521 Final Project/denoiser")
    ds = load_dataset("detection-datasets/coco", split="train[:3000]")
    ds.save_to_disk(DATASET_DIR)

ds = load_or_create_coco_dataset()

#ds = load_dataset("detection-datasets/coco", split="train[:3000]")

splits = ds.train_test_split(test_size=0.2, seed=42)
train = splits["train"]
temp = splits["test"]

# Split temp into val and test (50/50)
val_test = temp.train_test_split(test_size=0.5, seed=42)
val = val_test["train"]
test = val_test["test"]

📂 Loading dataset from disk: /content/drive/MyDrive/AAI-521 Final Project/super_res


In [ ]:
train

Dataset({
    features: ['image_id', 'image', 'width', 'height', 'objects'],
    num_rows: 2400
})

In [ ]:
test

Dataset({
    features: ['image_id', 'image', 'width', 'height', 'objects'],
    num_rows: 300
})

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms as T
import random
from PIL import Image


class CocoDenoiseHF(Dataset):
    def __init__(self, hf_dataset, resolution=256, noise_std=0.3):
        self.ds = hf_dataset
        self.noise_std = noise_std

        self.transform = T.Compose([
            T.Resize((resolution, resolution)),
            T.ToTensor(),
            T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Scale to [-1, 1] for Stable Diffusion
        ])

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]

        # Use the image directly
        img = item["image"]

        # Convert to RGB if grayscale
        if img.mode != 'RGB':
            img = img.convert('RGB')

        img = self.transform(img)

        # Create noisy version (add noise in [-1, 1] space)
        noise = torch.randn_like(img) * self.noise_std
        noisy = (img + noise).clamp(-1, 1)  # Clamp to [-1, 1] since we normalized

        return {
            "pixel_values": noisy,  # What the training script expects
            "clean_image": img      # Optional: keep for reference
        }

In [ ]:
# For super resolution
class CocoSuperResHF(Dataset):
    """
    COCO dataset for Super-Resolution fine-tuning.
    Produces:
        - 'lr_image'  : upsampled low-res (model input)
        - 'hr_image'  : high-res ground truth (target)
    """

    def __init__(self, hf_dataset, hr_resolution=256, scale_factor=4):
        self.ds = hf_dataset
        self.scale = scale_factor
        self.hr_resolution = hr_resolution

        self.hr_transform = T.Compose([
            T.Resize((hr_resolution, hr_resolution), interpolation=Image.BICUBIC),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])

        # low-res is hr_resolution / scale, then upsampled back to hr_resolution
        self.lr_down = T.Resize((hr_resolution // scale_factor,
                                 hr_resolution // scale_factor),
                                 interpolation=Image.BICUBIC)

        self.lr_up = T.Resize((hr_resolution, hr_resolution),
                              interpolation=Image.BICUBIC)

        self.to_norm = T.Compose([
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item["image"]

        if img.mode != "RGB":
            img = img.convert("RGB")

        # High-res clean image
        hr = self.hr_transform(img)

        # Create low-res → upsampled image
        lr_small = self.lr_down(img)
        lr_upsampled = self.lr_up(lr_small)

        # Normalize LR to match UNet input format [-1,1]
        lr = self.to_norm(lr_upsampled)

        return {
            "pixel_values": lr,      # Model input
            "clean_image": hr        # Target latent for noise prediction loss
        }

In [ ]:
import torch
import gc

# Clear all GPU memory
torch.cuda.empty_cache()
gc.collect()


60

In [ ]:
# fine_tune_super_resolution.py
import torch
import torchvision.transforms as T
from torch import nn
from torch.utils.data import DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from transformers import CLIPTokenizer, CLIPTextModel
from accelerate import Accelerator
import random
import matplotlib.pyplot as plt
import torch.nn.functional as F

loss_history = []

def smooth_curve(values, alpha=0.9):
    smoothed = []
    last = values[0]
    for v in values:
        last = alpha * last + (1 - alpha) * v
        smoothed.append(last)
    return smoothed

def main():
    accelerator = Accelerator(mixed_precision="fp16")

    # ----- Load pretrained SD Upscaler components -----
    model_name = "stabilityai/stable-diffusion-x4-upscaler"

    accelerator.print(f"Loading super-resolution model: {model_name}")

    vae = AutoencoderKL.from_pretrained(model_name, subfolder="vae")
    text_encoder = CLIPTextModel.from_pretrained(model_name, subfolder="text_encoder")
    tokenizer = CLIPTokenizer.from_pretrained(model_name, subfolder="tokenizer")
    unet = UNet2DConditionModel.from_pretrained(model_name, subfolder="unet")
    noise_scheduler = DDPMScheduler.from_pretrained(model_name, subfolder="scheduler")

    # Enable gradient checkpointing
    unet.enable_gradient_checkpointing()

    # Freeze VAE + text encoder (only train UNet)
    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)

    # ----- Dataset -----
    ds = CocoSuperResHF(train)
    dl = DataLoader(ds, batch_size=8, shuffle=True, num_workers=8)

    # Optimizer
    optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-5)

    unet, optimizer, dl, vae, text_encoder = accelerator.prepare(
        unet, optimizer, dl, vae, text_encoder
    )

    # Training
    num_epochs = 5
    vae_scale = 0.18215

    for epoch in range(num_epochs):
        for step, batch in enumerate(dl):
            # 1 — Encode target high-res image to latent
            with torch.no_grad():
                latents = vae.encode(batch["clean_image"]).latent_dist.sample()
                latents *= vae_scale

            # 2 — Sample random diffusion timestep
            bsz = latents.shape[0]
            timesteps = torch.randint(
                0,
                noise_scheduler.config.num_train_timesteps,
                (bsz,),
                device=latents.device
            ).long()

            # 3 — Add noise
            noise = torch.randn_like(latents)
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # 4 — Text conditioning (empty for unconditional)
            tokens = tokenizer(
                [""] * bsz,
                padding="max_length",
                max_length=tokenizer.model_max_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.to(latents.device)

            with torch.no_grad():
                text_embeds = text_encoder(tokens).last_hidden_state

            # 5 — Prepare low-res conditioning
            # Get low-res RGB image and resize to match latent spatial dimensions
            low_res_rgb = batch["pixel_values"]  # [B, 3, 256, 256]

            # Resize to match latent spatial size (64x64)
            low_res_rgb_resized = F.interpolate(
                low_res_rgb,
                size=(noisy_latents.shape[2], noisy_latents.shape[3]),
                mode='bilinear',
                align_corners=False
            )

            # Sample noise level for conditioning (0-1000)
            noise_level = torch.randint(0, 1000, (bsz,), device=latents.device).long()

            # Debug on first iteration
            if step == 0 and epoch == 0:
                accelerator.print(f"\n=== DEBUG: First batch ===")
                accelerator.print(f"Low-res input shape: {batch['pixel_values'].shape}")
                accelerator.print(f"High-res target shape: {batch['clean_image'].shape}")
                accelerator.print(f"Noisy latents shape: {noisy_latents.shape}")
                accelerator.print(f"Low-res RGB resized shape: {low_res_rgb_resized.shape}")
                accelerator.print(f"Noise level shape: {noise_level.shape}")
                accelerator.print(f"Text embeds shape: {text_embeds.shape}")
                accelerator.print(f"=== End DEBUG ===\n")

            # Concatenate: [B, 4, H, W] + [B, 3, H, W] = [B, 7, H, W]
            concat_latents = torch.cat([noisy_latents, low_res_rgb_resized], dim=1)

            # 6 — Predict noise
            noise_pred = unet(
                concat_latents,
                timesteps,
                encoder_hidden_states=text_embeds,
                class_labels=noise_level
            ).sample

            # 7 — Compute loss
            loss = nn.functional.mse_loss(noise_pred, noise)

            if accelerator.is_main_process:
                loss_history.append(loss.item())

            # Backprop
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()

            if step % 50 == 0:
                accelerator.print(f"Epoch {epoch} | Step {step} | Loss {loss.item():.4f}")

        accelerator.wait_for_everyone()
        accelerator.save_state(f"checkpoint-superres-epoch-{epoch}")

        # Plot loss curve
        if accelerator.is_main_process:
            plt.figure(figsize=(10, 5))
            plt.plot(loss_history, label="Raw Loss", alpha=0.3)
            plt.plot(smooth_curve(loss_history), label="Smoothed Loss (EMA)")
            plt.title("Super-Resolution Training Loss")
            plt.xlabel("Step")
            plt.ylabel("Loss")
            plt.grid(True)
            plt.legend()
            plt.savefig(f"loss_curve_superres_epoch_{epoch}.png")
            plt.close()

            accelerator.print(f"Saved loss curve to loss_curve_superres_epoch_{epoch}.png")

    # Save final model
    if accelerator.is_main_process:
        accelerator.print("Saving fine-tuned super-resolution model...")

        output_dir = f"{DATASET_DIR}/sd_finetuned_super_res"
        os.makedirs(output_dir, exist_ok=True)

        # Unwrap model before saving
        unwrapped_unet = accelerator.unwrap_model(unet)
        unwrapped_unet.save_pretrained(f"{output_dir}/unet")
        noise_scheduler.save_pretrained(f"{output_dir}/scheduler")

        accelerator.print(f"Model saved to: {output_dir}")
        accelerator.print("Training complete!")

if __name__ == "__main__":
    main()

Loading super-resolution model: stabilityai/stable-diffusion-x4-upscaler

=== DEBUG: First batch ===
Low-res input shape: torch.Size([8, 3, 256, 256])
High-res target shape: torch.Size([8, 3, 256, 256])
Noisy latents shape: torch.Size([8, 4, 64, 64])
Low-res RGB resized shape: torch.Size([8, 3, 64, 64])
Noise level shape: torch.Size([8])
Text embeds shape: torch.Size([8, 77, 1024])
=== End DEBUG ===

Epoch 0 | Step 0 | Loss 1.0852
Epoch 0 | Step 50 | Loss 0.4290
Epoch 0 | Step 100 | Loss 0.1921
Epoch 0 | Step 150 | Loss 0.1190
Epoch 0 | Step 200 | Loss 0.2568
Epoch 0 | Step 250 | Loss 0.2481
Saved loss curve to loss_curve_superres_epoch_0.png
Epoch 1 | Step 0 | Loss 0.2542
Epoch 1 | Step 50 | Loss 0.1341
Epoch 1 | Step 100 | Loss 0.1491
Epoch 1 | Step 150 | Loss 0.2636
Epoch 1 | Step 200 | Loss 0.1188
Epoch 1 | Step 250 | Loss 0.2333
Saved loss curve to loss_curve_superres_epoch_1.png
Epoch 2 | Step 0 | Loss 0.2039
Epoch 2 | Step 50 | Loss 0.2674
Epoch 2 | Step 100 | Loss 0.1245
Epoch 

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(loss_history, label="Raw Loss", alpha=0.3)
plt.plot(smooth_curve(loss_history), label="Smoothed Loss (EMA)")
plt.title("Training Loss Curve")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.savefig(f"loss_curve_smooth_superres.png")
plt.close()

In [1]:
from huggingface_hub import login
login()

In [ ]:
# For running metrics this is the diffuser pipeline.
#model_id = "runwayml/stable-diffusion-v1-5"
model_id = "stabilityai/stable-diffusion-x4-upscaler"
from diffusers import StableDiffusionUpscalePipeline, UNet2DConditionModel, AutoencoderKL, DDPMScheduler
import torch

print(f"Loading diffusion pipeline: {model_id}")

denoise_pipe = StableDiffusionUpscalePipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    safety_checker=None   # often needed for img2img in local environments
)

# Custom fine tuned model
DATASET_DIR = "/content/drive/MyDrive/AAI-521 Final Project/denoiser/sd_finetuned_denoiser/unet" if TRAIN_DENOISER else "/content/drive/MyDrive/AAI-521 Final Project/super_res/sd_finetuned_super_res/unet"
print("Loading fine-tuned UNet...")
fine_tuned_unet = UNet2DConditionModel.from_pretrained(
    DATASET_DIR,
    torch_dtype=denoise_pipe.unet.dtype,
)


Loading diffusion pipeline: stabilityai/stable-diffusion-x4-upscaler


model_index.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading fine-tuned UNet...


In [2]:
print("Pushing unet to huggingface")
fine_tuned_unet.push_to_hub("dydsa/superres_unet", token=hf_key)

In [ ]:
#
denoise_pipe.unet = fine_tuned_unet
print("Fine-tuned UNet loaded.")

denoise_pipe = denoise_pipe.to("cuda" if torch.cuda.is_available() else "cpu")

def diffusion_denoise(
    pil_image,
    prompt="super resolution",
    negative_prompt="grain, noise, artifacts, blur",
    denoise_strength=0.25,        # Lower = closer to original; higher = stronger denoising
    guidance_scale=7.0,
    num_inference_steps=25
):
    """
    Perform diffusion-based denoising using Stable Diffusion img2img.
    The image is slightly "nudged" toward a clean version of itself.
    """
    print("Running diffusion pipeline...")

    # Ensure correct size for the diffusion pipeline
    image_resized = pil_image.resize(IMG_SIZE)

    result = denoise_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image_resized,
        strength=denoise_strength,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    )

    denoised_image = result.images[0]

    print("Pipeline complete.")
    return denoised_image


Fine-tuned UNet loaded.


In [ ]:
# --- Helper functions (ensuring all dependencies are met in this block) ---

def _device():
    return "cuda" if torch.cuda.is_available() else "cpu"

def pil_to_tensor(img: Image.Image) -> torch.Tensor:
    """Convert PIL → C×H×W float tensor in [0, 1] on the evaluation device."""
    arr = np.array(img).astype(np.float32) / 255.0          # H×W×C, 0–1
    tensor = torch.from_numpy(arr).permute(2, 0, 1)        # C×H×W
    return tensor.to(_device())

def compute_metrics(
    clean: torch.Tensor,
    denoised: torch.Tensor,
) -> Dict[str, float]:
    """
    Compute a dictionary of metrics for a *single* image pair.
    All tensors must be on the same device and have shape (C, H, W) in [0,1].
    """
    # Ensure the tensors are 4–D (batch dim) for torchmetrics
    clean = clean.unsqueeze(0)
    denoised = denoised.unsqueeze(0)

    # PSNR (higher = better) - Updated API
    from torchmetrics.image import PeakSignalNoiseRatio
    psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(_device())
    psnr = psnr_metric(denoised, clean).item()

    # SSIM (higher = better) - Updated API
    from torchmetrics.image import StructuralSimilarityIndexMeasure
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(_device())
    ssim = ssim_metric(denoised, clean).item()

    # LPIPS (lower = better) – Updated API for piq
    from piq import LPIPS
    lpips_metric = LPIPS(reduction='mean').to(_device())
    # piq LPIPS expects inputs in [0, 1] range
    lpips = lpips_metric(denoised, clean).item()

    return {"psnr": psnr, "ssim": ssim, "lpips": lpips}

def tensor_to_pil(tensor_img):
    """Converts a C x H x W tensor in [-1, 1] to a PIL Image."""
    # Denormalize from [-1, 1] to [0, 1]
    tensor_img = (tensor_img / 2 + 0.5).clamp(0, 1)
    # Convert to H x W x C, then to numpy, then to PIL
    np_img = tensor_img.permute(1, 2, 0).cpu().numpy()
    np_img = (np_img * 255).astype(np.uint8)
    return Image.fromarray(np_img)


# --- Evaluation Setup ---

IMG_SIZE = (256, 256) # Based on model resolution set during training

# Alias diffusion_denoise from previous cell
denoise_image = diffusion_denoise

# Denoise keyword arguments
denoise_kwargs = {
    "prompt": "", # Empty prompt for fine tuned
    "negative_prompt": "grain, noise, artifacts, blur, bad quality, ugly, watermark",
    "denoise_strength": 0.3, # This can be tuned
    "guidance_scale": 7.5,
    "num_inference_steps": 25
}

# 1. Create dataset and dataloader for the test split
test_dataset = CocoDenoiseHF(test, resolution=IMG_SIZE[0])
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2) # Batch size 1 for evaluation

results = []

print("Starting evaluation on the test set...")
for i, batch in enumerate(tqdm(test_dataloader, desc="Evaluating Test Set")):
    # Move tensors to the correct device
    noisy_tensor = batch["pixel_values"].to(_device())
    clean_tensor_target = batch["clean_image"].to(_device()) # This is the ground truth clean image

    # Convert noisy tensor to PIL for diffusion_denoise (assuming batch size is 1 for evaluation)
    noisy_pil = tensor_to_pil(noisy_tensor[0])

    # -------------------------------------------------
    # 2‼️ Denoise (re‑uses the shared pipeline)
    # -------------------------------------------------
    denoised_pil = denoise_image(noisy_pil, **denoise_kwargs)

    # -------------------------------------------------
    # 3‼️ Convert to torch tensors for metrics
    # -------------------------------------------------
    # Convert clean_tensor_target from [-1, 1] to [0, 1] for compute_metrics
    clean_tensor_for_metrics = (clean_tensor_target[0] / 2 + 0.5).clamp(0, 1)

    # Convert denoised PIL image to tensor in [0, 1]
    denoised_tensor_for_metrics = pil_to_tensor(denoised_pil)

    # -------------------------------------------------
    # 4‼️ Compute metrics
    # -------------------------------------------------
    metric_dict = compute_metrics(clean_tensor_for_metrics, denoised_tensor_for_metrics)
    metric_dict["index"] = i # Use index as identifier if no filename is available
    results.append(metric_dict)

print("\nEvaluation complete. Aggregating results...")

# Aggregate and print results
if results:
    avg_psnr = sum(r["psnr"] for r in results) / len(results)
    avg_ssim = sum(r["ssim"] for r in results) / len(results)
    avg_lpips = sum(r["lpips"] for r in results) / len(results)
    print(f"Average PSNR: {avg_psnr:.4f}")
    print(f"Average SSIM: {avg_ssim:.4f}")
    print(f"Average LPIPS: {avg_lpips:.4f}")
else:
    print("No results to aggregate.")

# Denoiser
#Evaluation complete. Fine tuned results...
# Test set is 300 images
#Average PSNR: 15.7459 (higher is better)
#Average SSIM: 0.1540. (higher is better)
#Average LPIPS: 0.6604 (lower is better)

# Now for the not fine tuned
#Evaluation complete. Aggregating results...
#Average PSNR: 12.7420 (higher is better)
#Average SSIM: 0.1270. (higher is better)
#Average LPIPS: 0.7009 (lower is better)

#Super res
#Fine tuned
#Evaluation complete. Aggregating results...
#Average PSNR: 15.3932
#Average SSIM: 0.1465
#Average LPIPS: 0.6754

# Standard
#Evaluation complete. Aggregating results...
#Average PSNR: 12.7745
#Average SSIM: 0.1275
#Average LPIPS: 0.7001